# Phase 1: 자율주행 VQA 데이터 파이프라인

## 목표
CARLA 시뮬레이터 데이터를 VLM 파인튜닝용 **QA 쌍**으로 자동 변환

## 핵심 아이디어
별도 레이블링 없이 기존 센서 데이터를 조합해 **자동으로 QA 생성**:
- YOLO labels → 객체 종류 + 위치
- Depth map → 객체까지 거리 (미터)
- Semantic map → 장면 구성 비율
- 규칙 기반 템플릿 → 다양한 QA 유형 생성

## 출력 형식
Qwen2-VL / LLaVA 파인튜닝 표준 포맷 (ShareGPT format)

In [ ]:
import matplotlib
matplotlib.rcParams['font.family'] = 'Malgun Gothic'
matplotlib.rcParams['axes.unicode_minus'] = False

import numpy as np
import json
import random
from pathlib import Path
from PIL import Image
from collections import defaultdict
import matplotlib.pyplot as plt
import matplotlib.patches as patches

random.seed(42)
np.random.seed(42)

# ── 경로 설정 ──────────────────────────────────────────────
DATA_DIR  = Path('../phase4_carla/data_collection/carla_dataset')
IMG_DIR   = DATA_DIR / 'images'
DEP_DIR   = DATA_DIR / 'depth'
SEM_DIR   = DATA_DIR / 'semantic'
LBL_DIR   = DATA_DIR / 'labels'
OUT_DIR   = Path('qa_dataset')
OUT_DIR.mkdir(exist_ok=True)

# 이미지 크기 (metadata.json 기준)
IMG_W, IMG_H = 1280, 720

img_files = sorted(IMG_DIR.glob('*.jpg'))
print(f'총 프레임: {len(img_files)}개')

## 1. 클래스 매핑 정의

In [ ]:
# YOLO 클래스 (carla.yaml 기준)
YOLO_CLASSES = {
    0: '보행자',
    1: '자전거',
    2: '차량',
    3: '오토바이',
    5: '버스',
    7: '트럭',
}

# CARLA Semantic 클래스
SEM_CLASSES = {
    1: '건물',   2: '펜스',     3: '기타',      4: '보행자',
    5: '기둥',   6: '차선',     7: '도로',      8: '인도',
    9: '식물',  10: '차량',    11: '벽',       12: '신호등',
   13: '하늘',  20: '동적요소', 22: '지형',     24: '가드레일',
}

# 위험도 판단 기준 (거리 미터 기준)
DANGER_THRESHOLDS = {
    '즉각위험':  5.0,
    '주의필요': 15.0,
    '인지필요': 30.0,
}

print('클래스 매핑 정의 완료')

## 2. 프레임 파싱 — 구조화된 장면 정보 추출

In [ ]:
def parse_frame(img_path: Path) -> dict:
    """
    하나의 프레임에서 QA 생성에 필요한 구조화된 정보를 추출.

    Returns:
        {
          'frame_id': str,
          'objects': [{'class': str, 'dist_m': float, 'cx': float, 'cy': float,
                        'w': float, 'h': float, 'position': str}],
          'scene': {'road_pct': float, 'sidewalk_pct': float, ...},
          'nearest': {'class': str, 'dist_m': float} or None,
          'danger_level': str,
          'has_pedestrian': bool,
          'has_vehicle': bool,
        }
    """
    stem = img_path.stem

    # 1) YOLO 라벨 파싱
    lbl_path = LBL_DIR / f'{stem}.txt'
    objects = []
    if lbl_path.exists():
        depth = np.load(str(DEP_DIR / f'{stem}.npy'))
        with open(lbl_path) as f:
            for line in f:
                parts = line.strip().split()
                if not parts:
                    continue
                cls_id, cx, cy, w, h = int(parts[0]), *map(float, parts[1:5])
                if cls_id not in YOLO_CLASSES:
                    continue

                # 픽셀 좌표로 변환해 depth 조회
                px = int(np.clip(cx * IMG_W, 0, IMG_W - 1))
                py = int(np.clip(cy * IMG_H, 0, IMG_H - 1))
                # bbox 내 depth 중앙값 (단일 픽셀보다 안정적)
                x1 = max(0, int((cx - w/2) * IMG_W))
                x2 = min(IMG_W, int((cx + w/2) * IMG_W))
                y1 = max(0, int((cy - h/2) * IMG_H))
                y2 = min(IMG_H, int((cy + h/2) * IMG_H))
                roi = depth[y1:y2, x1:x2]
                dist_m = float(np.median(roi[roi < 500])) if roi.size > 0 else depth[py, px]

                # 수평 위치 판단
                if cx < 0.33:
                    pos = '좌측'
                elif cx > 0.67:
                    pos = '우측'
                else:
                    pos = '전방'

                objects.append({
                    'class': YOLO_CLASSES[cls_id],
                    'dist_m': round(dist_m, 1),
                    'cx': cx, 'cy': cy,
                    'w': w, 'h': h,
                    'position': pos,
                })

    objects.sort(key=lambda o: o['dist_m'])

    # 2) Semantic 장면 구성 분석
    sem = np.array(Image.open(str(SEM_DIR / f'{stem}.png')))
    total = sem.size
    scene = {}
    for cid, cname in SEM_CLASSES.items():
        pct = float((sem == cid).sum() / total * 100)
        if pct > 0.1:
            scene[cname] = round(pct, 1)

    # 3) 위험도 판단
    nearest = objects[0] if objects else None
    if nearest is None:
        danger = '없음'
    elif nearest['dist_m'] < DANGER_THRESHOLDS['즉각위험']:
        danger = '즉각위험'
    elif nearest['dist_m'] < DANGER_THRESHOLDS['주의필요']:
        danger = '주의필요'
    elif nearest['dist_m'] < DANGER_THRESHOLDS['인지필요']:
        danger = '인지필요'
    else:
        danger = '안전'

    return {
        'frame_id':      stem,
        'objects':       objects,
        'scene':         scene,
        'nearest':       nearest,
        'danger_level':  danger,
        'has_pedestrian': any(o['class'] == '보행자' for o in objects),
        'has_vehicle':    any(o['class'] in ('차량', '버스', '트럭') for o in objects),
    }

# 테스트
sample = parse_frame(img_files[0])
print(json.dumps(sample, ensure_ascii=False, indent=2))

## 3. QA 생성기 — 7가지 유형

In [ ]:
def gen_qa_scene_description(info: dict) -> dict | None:
    """QA 유형 1: 장면 전체 설명"""
    objs = info['objects']
    scene = info['scene']

    scene_parts = []
    if '도로' in scene:
        scene_parts.append(f"도로({scene['도로']:.0f}%)")
    if '건물' in scene:
        scene_parts.append(f"건물({scene['건물']:.0f}%)")
    if '식물' in scene:
        scene_parts.append(f"식물({scene['식물']:.0f}%)")
    scene_str = ', '.join(scene_parts) if scene_parts else '도시 환경'

    if not objs:
        obj_str = '주변에 차량이나 보행자가 없습니다'
    else:
        parts = []
        for o in objs[:3]:
            parts.append(f"{o['position']} {o['dist_m']}m 거리에 {o['class']}")
        obj_str = ', '.join(parts)

    q = '이 자율주행 차량 카메라 영상을 설명해주세요.'
    a = f'도시 도로 주행 장면입니다. 장면 구성: {scene_str}. {obj_str}.'
    return {'q': q, 'a': a, 'type': 'scene_description'}


def gen_qa_nearest_object(info: dict) -> dict | None:
    """QA 유형 2: 가장 가까운 객체"""
    if not info['nearest']:
        return None
    n = info['nearest']
    q = '카메라에서 가장 가까운 객체는 무엇이고 거리는 얼마입니까?'
    a = f'가장 가까운 객체는 {n["position"]}에 있는 {n["class"]}이며, 약 {n["dist_m"]}m 거리에 있습니다.'
    return {'q': q, 'a': a, 'type': 'nearest_object'}


def gen_qa_danger_assessment(info: dict) -> dict | None:
    """QA 유형 3: 위험도 평가"""
    level = info['danger_level']
    n = info['nearest']

    # n이 None일 수 있으므로 dict 리터럴 대신 if/elif 사용
    if level == '없음':
        yn, reason = '아니오', '현재 장면에 차량이나 보행자가 감지되지 않았습니다. 안전한 상황입니다.'
    elif level == '즉각위험':
        yn, reason = '예', f'{n["position"]} {n["dist_m"]}m 이내에 {n["class"]}이 있어 즉각적인 제동이 필요합니다.'
    elif level == '주의필요':
        yn, reason = '예', f'{n["position"]} {n["dist_m"]}m 거리에 {n["class"]}이 있어 속도를 줄이고 주의가 필요합니다.'
    elif level == '인지필요':
        yn, reason = '주의', f'{n["position"]} {n["dist_m"]}m 거리에 {n["class"]}이 있습니다. 인지하고 모니터링하세요.'
    else:  # 안전
        yn, reason = '아니오', f'가장 가까운 {n["class"]}이 {n["dist_m"]}m로 충분한 거리를 유지하고 있습니다.'

    q = '현재 주행 상황에서 즉각적인 위험 요소가 있습니까?'
    a = f'{yn}. {reason}'
    return {'q': q, 'a': a, 'type': 'danger_assessment'}


def gen_qa_action_recommendation(info: dict) -> dict | None:
    """QA 유형 4: 행동 추천"""
    level = info['danger_level']
    n = info['nearest']

    # n이 None일 수 있으므로 dict 리터럴 대신 if/elif 사용
    if level == '없음':
        action = '현재 주행 속도를 유지하세요. 감지된 장애물이 없습니다.'
    elif level == '즉각위험':
        action = f'즉시 감속 또는 제동하세요. {n["position"]}에 {n["class"]}이 {n["dist_m"]}m 이내에 있습니다.'
    elif level == '주의필요':
        action = f'속도를 줄이고 {n["position"]}의 {n["class"]}({n["dist_m"]}m)를 주시하세요.'
    elif level == '인지필요':
        action = f'현재 속도를 유지하되 {n["position"]}의 {n["class"]}({n["dist_m"]}m)를 계속 모니터링하세요.'
    else:  # 안전
        action = f'현재 속도를 유지하세요. 가장 가까운 객체({n["class"]})가 {n["dist_m"]}m로 안전 거리를 확보하고 있습니다.'

    q = '현재 상황에서 자율주행 시스템이 취해야 할 행동을 알려주세요.'
    return {'q': q, 'a': action, 'type': 'action_recommendation'}


def gen_qa_pedestrian(info: dict) -> dict | None:
    """QA 유형 5: 보행자 존재 여부"""
    peds = [o for o in info['objects'] if o['class'] == '보행자']
    q = '이 장면에 보행자가 있습니까? 있다면 위치와 거리를 알려주세요.'
    if not peds:
        a = '현재 장면에 보행자가 감지되지 않았습니다.'
    elif len(peds) == 1:
        p = peds[0]
        a = f'보행자 1명이 {p["position"]} {p["dist_m"]}m 거리에 있습니다.'
    else:
        parts = [f'{p["position"]} {p["dist_m"]}m' for p in peds]
        a = f'보행자 {len(peds)}명이 감지되었습니다: {", ".join(parts)}.'
    return {'q': q, 'a': a, 'type': 'pedestrian_check'}


def gen_qa_object_count(info: dict) -> dict | None:
    """QA 유형 6: 객체 개수"""
    if not info['objects']:
        return None
    from collections import Counter
    cnt = Counter(o['class'] for o in info['objects'])
    parts = [f'{cls} {n}개' for cls, n in cnt.items()]
    q = '이 장면에서 감지된 객체의 종류와 수를 알려주세요.'
    a = f'총 {len(info["objects"])}개의 객체가 감지되었습니다: {", ".join(parts)}.'
    return {'q': q, 'a': a, 'type': 'object_count'}


def gen_qa_safety_distance(info: dict) -> dict | None:
    """QA 유형 7: 안전 거리 판단"""
    if not info['nearest']:
        return None
    n = info['nearest']
    safe = n['dist_m'] >= 14.0
    q = f'{n["position"]}의 {n["class"]}까지 안전 거리가 확보되어 있습니까?'
    if safe:
        a = f'예. {n["class"]}까지 {n["dist_m"]}m로 권장 안전 거리(14m 이상)가 확보되어 있습니다.'
    else:
        a = f'아니오. {n["class"]}까지 {n["dist_m"]}m로 권장 안전 거리(14m)보다 가깝습니다. 감속이 필요합니다.'
    return {'q': q, 'a': a, 'type': 'safety_distance'}


QA_GENERATORS = [
    gen_qa_scene_description,
    gen_qa_nearest_object,
    gen_qa_danger_assessment,
    gen_qa_action_recommendation,
    gen_qa_pedestrian,
    gen_qa_object_count,
    gen_qa_safety_distance,
]

# 테스트
test_info = parse_frame(img_files[5])  # 객체 있는 프레임
print(f'프레임 {test_info["frame_id"]}: 객체 {len(test_info["objects"])}개, 위험도={test_info["danger_level"]}')
print()
for gen in QA_GENERATORS:
    qa = gen(test_info)
    if qa:
        print(f'[{qa["type"]}]')
        print(f'  Q: {qa["q"]}')
        print(f'  A: {qa["a"]}')
        print()

## 4. 전체 데이터셋 생성

In [ ]:
def frame_to_sharegpt(img_path: Path, qa: dict) -> dict:
    """QA 쌍 → ShareGPT 포맷 (Qwen2-VL / LLaVA 파인튜닝 표준)"""
    return {
        'id': f"{img_path.stem}_{qa['type']}",
        'image': str(img_path.resolve()),
        'conversations': [
            {'from': 'human', 'value': f'<image>\n{qa["q"]}'},
            {'from': 'gpt',   'value': qa['a']},
        ],
        'metadata': {'qa_type': qa['type'], 'frame_id': img_path.stem}
    }


# 전체 프레임 처리
all_samples = []
type_counter = defaultdict(int)
skipped = 0

for img_f in img_files:
    try:
        info = parse_frame(img_f)
    except Exception as e:
        skipped += 1
        continue

    for gen in QA_GENERATORS:
        qa = gen(info)
        if qa:
            sample = frame_to_sharegpt(img_f, qa)
            all_samples.append(sample)
            type_counter[qa['type']] += 1

print(f'총 QA 쌍: {len(all_samples)}개 (스킵: {skipped}프레임)')
print('\nQA 유형별 분포:')
for t, c in sorted(type_counter.items()):
    print(f'  {t:<25}: {c:>4}개')

## 5. Train / Val 분할 & 저장

In [ ]:
# 프레임 ID 기준으로 분할 (QA 유형이 같은 프레임이 train/val에 동시에 들어가지 않도록)
all_frame_ids = sorted(set(s['metadata']['frame_id'] for s in all_samples))
random.shuffle(all_frame_ids)

split_idx = int(len(all_frame_ids) * 0.85)
train_ids = set(all_frame_ids[:split_idx])
val_ids   = set(all_frame_ids[split_idx:])

train_samples = [s for s in all_samples if s['metadata']['frame_id'] in train_ids]
val_samples   = [s for s in all_samples if s['metadata']['frame_id'] in val_ids]

# 저장
with open(OUT_DIR / 'train.json', 'w', encoding='utf-8') as f:
    json.dump(train_samples, f, ensure_ascii=False, indent=2)

with open(OUT_DIR / 'val.json', 'w', encoding='utf-8') as f:
    json.dump(val_samples, f, ensure_ascii=False, indent=2)

print(f'Train: {len(train_samples)}개 ({len(train_ids)}프레임)')
print(f'Val  : {len(val_samples)}개 ({len(val_ids)}프레임)')
print(f'저장 완료: {OUT_DIR}/train.json, val.json')

## 6. 데이터 품질 검증 시각화

In [ ]:
# QA 유형 분포 pie chart
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle('VQA 데이터셋 구성', fontsize=14, fontweight='bold')

labels = list(type_counter.keys())
values = list(type_counter.values())
axes[0].pie(values, labels=labels, autopct='%1.1f%%', startangle=140)
axes[0].set_title(f'QA 유형 분포 (총 {len(all_samples)}개)')

# 위험도 분포
danger_counts = defaultdict(int)
for img_f in img_files:
    try:
        info = parse_frame(img_f)
        danger_counts[info['danger_level']] += 1
    except:
        pass

danger_labels = list(danger_counts.keys())
danger_vals   = list(danger_counts.values())
colors = {'즉각위험': '#d9534f', '주의필요': '#f0ad4e',
          '인지필요': '#5bc0de', '안전': '#5cb85c', '없음': '#aaaaaa'}
bar_colors = [colors.get(l, '#999999') for l in danger_labels]
bars = axes[1].bar(danger_labels, danger_vals, color=bar_colors, edgecolor='black')
axes[1].set_title('프레임별 위험도 분포')
axes[1].set_ylabel('프레임 수')
for bar, val in zip(bars, danger_vals):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 2,
                 str(val), ha='center', fontsize=10)

plt.tight_layout()
plt.savefig('qa_dataset_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print('분포 차트 저장: qa_dataset_distribution.png')

## 7. 샘플 시각화 — QA 쌍이 이미지와 일치하는지 확인

In [ ]:
# 객체 있는 프레임 4개 샘플링
frames_with_objs = []
for img_f in img_files:
    try:
        info = parse_frame(img_f)
        if info['objects']:
            frames_with_objs.append((img_f, info))
    except:
        pass

sample_frames = random.sample(frames_with_objs, min(4, len(frames_with_objs)))

fig, axes = plt.subplots(2, 2, figsize=(14, 8))
fig.suptitle('QA 샘플 시각화 (bbox + 생성된 QA)', fontsize=13)

for ax, (img_f, info) in zip(axes.flatten(), sample_frames):
    img = np.array(Image.open(img_f))
    ax.imshow(img)

    # bbox 그리기
    for obj in info['objects']:
        x1 = (obj['cx'] - obj['w']/2) * IMG_W
        y1 = (obj['cy'] - obj['h']/2) * IMG_H
        bw = obj['w'] * IMG_W
        bh = obj['h'] * IMG_H
        color = 'red' if obj['class'] == '보행자' else 'cyan'
        rect = patches.Rectangle((x1, y1), bw, bh,
                                   linewidth=2, edgecolor=color, facecolor='none')
        ax.add_patch(rect)
        ax.text(x1, y1 - 5, f"{obj['class']} {obj['dist_m']}m",
                color=color, fontsize=8, fontweight='bold')

    # 생성된 QA 중 하나 표시
    qa = gen_qa_danger_assessment(info)
    if qa:
        ax.set_title(f"[{info['frame_id']}] 위험도: {info['danger_level']}\nQ: {qa['q'][:30]}...\nA: {qa['a'][:50]}...",
                     fontsize=7)
    ax.axis('off')

plt.tight_layout()
plt.savefig('qa_sample_visualization.png', dpi=150, bbox_inches='tight')
plt.show()
print('샘플 시각화 저장: qa_sample_visualization.png')

## 8. 데이터셋 통계 요약

In [ ]:
# 답변 길이 분포 확인 (너무 짧거나 길면 품질 문제)
answer_lengths = [len(s['conversations'][1]['value']) for s in all_samples]

print('=' * 50)
print('  VQA 데이터셋 통계')
print('=' * 50)
print(f'  총 QA 쌍     : {len(all_samples)}개')
print(f'  Train        : {len(train_samples)}개')
print(f'  Val          : {len(val_samples)}개')
print(f'  프레임 수    : {len(img_files)}개')
print(f'  QA 유형      : {len(type_counter)}가지')
print()
print(f'  답변 길이 (문자 수):')
print(f'    평균  : {np.mean(answer_lengths):.0f}')
print(f'    최소  : {np.min(answer_lengths)}')
print(f'    최대  : {np.max(answer_lengths)}')
print()
print('  저장 파일:')
print(f'    qa_dataset/train.json')
print(f'    qa_dataset/val.json')
print('=' * 50)
print()
print('  다음 단계: 02_vlm_finetune.ipynb')
print('  Qwen2-VL-7B QLoRA 파인튜닝')